# Chapter 16 Exercises: AI, Machine Learning, and Text in Finance

This notebook contains coding exercises for Chapter 16.
Students will work with financial text data, implement a simple bag-of-words
classifier, and compare it to a transformer-based sentiment model.

> **Status:** Placeholder — exercises to be developed alongside `/draft-exercises`.

## Data Lab -- SEC EDGAR

Combine textual signals (uncertainty, readability) with XBRL fundamentals to build a cross-sectional return predictor.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()
UNCERTAINTY_WORDS = {"uncertain","uncertainty","unpredictable","volatility","fluctuate",
                     "unclear","approximate","depend","contingent","may","might","could","estimate"}
LM_NEG = {"risk","loss","uncertain","decline","adverse","default","impair","volatile",
           "litigation","breach","failed","reduced","weakness","debt","distress",
           "penalty","violation","exposure","downturn","writedown"}
LM_POS = {"profitable","strong","growth","improved","exceeded","innovative","efficient",
          "leading","record","gained","successful","robust","increased","advancing",
          "outperformed","benefit","enhanced","positive","achieved","favourable"}

### Exercise [B]: Uncertainty Word Frequency Trend

In [ ]:
# --- Exercise [B]: Uncertainty Word Frequency Trend ---
cik_msft = get_cik("MSFT")
subs_m   = get_submissions(cik_msft)
fm       = subs_m["filings"]["recent"]
k10_msft = [i for i, x in enumerate(fm["form"]) if x == "10-K"][:5]

VIX_ANNUAL = {2019:15.4, 2020:29.2, 2021:19.7, 2022:25.6, 2023:17.5}

rows_16b = []
for i in k10_msft:
    yr  = int(fm["filingDate"][i][:4])
    acc = fm["accessionNumber"][i].replace("-","")
    url = (f"https://www.sec.gov/Archives/edgar/data/{cik_msft.lstrip('0')}"
           f"/{acc}/{fm['primaryDocument'][i]}")
    time.sleep(0.15)
    html = requests.get(url, headers=EDGAR_HEADERS, timeout=30).text
    text = re.sub(r"<[^>]+>", " ", html).lower()
    words = re.findall(r"[a-z]+", text)
    n = max(len(words), 1)
    unc_freq = sum(1 for w in words if w in UNCERTAINTY_WORDS) * 1000 / n
    rows_16b.append({"year": yr, "unc_freq": unc_freq, "vix": VIX_ANNUAL.get(yr)})
    print(f"MSFT {yr}: unc={unc_freq:.2f}/1k words, VIX={VIX_ANNUAL.get(yr,'n/a')}")

df_16b = pd.DataFrame(rows_16b).sort_values("year")
fig, ax1 = plt.subplots(figsize=(9,5))
ax1.plot(df_16b["year"], df_16b["unc_freq"], "o-b", linewidth=2, label="Uncertainty words/1k")
ax1.set_xlabel("Year"); ax1.set_ylabel("Uncertainty frequency (per 1k words)", color="blue")
ax2 = ax1.twinx()
vix_rows = df_16b.dropna(subset=["vix"])
ax2.plot(vix_rows["year"], vix_rows["vix"], "s--r", linewidth=2, label="VIX (annual avg)")
ax2.set_ylabel("VIX", color="red")
ax1.set_title("MSFT 10-K Uncertainty Language vs. VIX")
fig.legend(loc="upper left", bbox_to_anchor=(0.1,0.9))
plt.tight_layout(); plt.show()

if df_16b["vix"].notna().sum() >= 3:
    corr = np.corrcoef(df_16b.dropna()["unc_freq"], df_16b.dropna()["vix"])[0,1]
    print(f"Pearson r (uncertainty vs VIX) = {corr:.3f}")

### Exercise [I]: Gunning Fog Index vs. Firm Size

In [ ]:
# --- Exercise [I]: Fog Index vs. Total Assets ---
TICKERS_16 = ["AAPL","MSFT","GOOGL","JPM","BAC","XOM","JNJ","GE","F","NFLX"]

def fog_index(text):
    words = text.split()
    W = max(len(words), 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    S = max(len(sents), 1)
    H = sum(1 for w in words if len(re.sub(r"[^a-z]","",w.lower())) >= 7)
    return 0.4 * (W/S + 100*H/W)

rows_16i = []
for t in TICKERS_16:
    try:
        cik = get_cik(t)
        html = fetch_10k_html(t)
        mda  = extract_mda(html)
        fog  = fog_index(mda)
        assets = get_annual_series(cik,"Assets").sort_values("date").iloc[-1]["Assets"]
        rows_16i.append({"ticker":t, "fog":fog, "log_assets":np.log10(max(assets,1))})
        print(f"{t}: Fog={fog:.1f}, log10(Assets)={np.log10(assets):.2f}")
    except Exception as e:
        print(f"{t}: {e}")

df_16i = pd.DataFrame(rows_16i)
if len(df_16i) >= 3:
    X = df_16i["log_assets"].values; y = df_16i["fog"].values
    Xb = np.c_[np.ones(len(X)), X]
    coef = np.linalg.lstsq(Xb, y, rcond=None)[0]
    print(f"\nOLS: Fog = {coef[0]:.2f} + {coef[1]:.2f} * log10(Assets)")
    fig, ax = plt.subplots(figsize=(7,5))
    ax.scatter(X, y, s=80, color="steelblue")
    for _, r in df_16i.iterrows(): ax.annotate(r["ticker"],(r["log_assets"],r["fog"]),fontsize=8)
    xline = np.linspace(X.min(), X.max(), 50)
    ax.plot(xline, coef[0]+coef[1]*xline, "r--", linewidth=1.5, label=f"OLS slope={coef[1]:.2f}")
    ax.set_xlabel("log10(Total Assets)"); ax.set_ylabel("Gunning Fog Index")
    ax.set_title("Readability vs. Firm Size"); ax.legend()
    plt.tight_layout(); plt.show()

### Exercise [A]: Text-and-Ratio Return Predictor

In [ ]:
# --- Exercise [A]: 6-Feature Return Predictor ---
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-20,20)))

feat_rows16, labels16 = [], []
for t in TICKERS_16:
    try:
        cik = get_cik(t)
        html = fetch_10k_html(t)
        mda  = extract_mda(html)
        words = re.findall(r"[a-z]+", mda.lower())
        n = max(len(words), 1)
        neg_freq = sum(1 for w in words if w in LM_NEG)/n
        unc_freq = sum(1 for w in words if w in UNCERTAINTY_WORDS)/n
        fog      = fog_index(mda)
        def gv(c):
            try: return float(get_annual_series(cik,c).sort_values("date").iloc[-1][c])
            except: return 0.0
        A  = gv("Assets")  or 1
        E  = gv("StockholdersEquity") or 1
        NI = gv("NetIncomeLoss")
        D  = gv("LongTermDebt")
        try:
            rev_s = get_annual_series(cik,"Revenues").sort_values("date")["Revenues"]
            rg = float(rev_s.iloc[-1]/rev_s.iloc[-2]-1) if len(rev_s)>=2 else 0
        except: rg=0
        feat_rows16.append([neg_freq, unc_freq, fog/100, D/A, NI/abs(E) if E else 0, rg])
        labels16.append(1 if t in ["AAPL","MSFT","GOOGL","JPM","JNJ","NFLX"] else 0)
    except Exception as e:
        print(f"{t}: {e}"); feat_rows16.append([0]*6); labels16.append(0)

X16 = np.array(feat_rows16); y16 = np.array(labels16, dtype=float)
X16 = (X16 - X16.mean(0)) / (X16.std(0)+1e-8)
Xb16 = np.c_[np.ones(len(X16)), X16]

# LOO-CV
hits = 0
for i in range(len(X16)):
    mask = np.arange(len(X16))!=i
    Xtr  = np.c_[np.ones(mask.sum()), X16[mask]]; ytr=y16[mask]
    w    = np.zeros(Xtr.shape[1])
    for _ in range(1000):
        p=sigmoid(Xtr@w); w-=0.3*Xtr.T@(p-ytr)/len(ytr)
    hits += int(int(sigmoid(np.r_[1,X16[i]]@w)>0.5)==int(y16[i]))

# Full model for coefficients
w_full = np.zeros(Xb16.shape[1])
for _ in range(2000):
    p=sigmoid(Xb16@w_full); w_full-=0.3*Xb16.T@(p-y16)/len(y16)

feat_names = ["lm_neg","uncertainty","fog/100","leverage","roe","rev_growth"]
print(f"LOO-CV accuracy: {hits}/{len(X16)} = {hits/len(X16):.2f}")
print("\nCoefficients:")
for name, coef in zip(["bias"]+feat_names, w_full):
    print(f"  {name:<15} {coef:+.4f}")
most_pred = feat_names[int(np.argmax(np.abs(w_full[1:])))]
print(f"\nMost predictive feature: {most_pred}")
print("\nNote: n=10 -- interpret with extreme caution (overfitting + multiple comparisons).")